<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/99_A_kaggle_diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Artefato A — Diagnóstico de Paridade do Protótipo Kaggle

Este notebook produz o diagnóstico do protótipo Kaggle refinado utilizado para comparar o denominador e a escala de `fail_rate` com o experimento FULL.

Seu papel é registrar a referência empírica de paridade antes da materialização do conjunto FULL, preservando a separação entre a camada de desenvolvimento metodológico e a validação em escala.

In [ ]:
# ============================================================
# NB99_A — Montagem do Google Drive e configuração dos caminhos
# Artefato A — Diagnóstico Kaggle de Paridade
# ============================================================

from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

def mount_google_drive(preferred_mount_point="/content/drive"):
    """
    Monta o Google Drive no Colab e retorna o caminho da raiz do MyDrive.
    Se o mountpoint padrão estiver problemático, tenta um mountpoint alternativo.
    """
    if not IN_COLAB:
        print("[drive] Ambiente não parece ser Colab. Montagem do Drive ignorada.")
        return None

    from google.colab import drive

    preferred = Path(preferred_mount_point)
    preferred_my_drive = preferred / "MyDrive"

    # Se já estiver montado corretamente, reutiliza.
    if preferred_my_drive.exists() and (preferred_my_drive / "Mestrado").exists():
        print(f"[drive] Google Drive já disponível em: {preferred_my_drive}")
        return preferred_my_drive

    # Tenta montagem padrão.
    try:
        print(f"[drive] Montando Google Drive em {preferred} ...")
        drive.mount(str(preferred), force_remount=False)
        if preferred_my_drive.exists():
            print(f"[drive] Google Drive montado em: {preferred_my_drive}")
            return preferred_my_drive
    except Exception as e:
        print(f"[drive] Falha na montagem padrão em {preferred}: {e}")

    # Fallback: usar outro ponto de montagem, útil se /content/drive foi poluído por execução anterior.
    fallback = Path("/content/gdrive")
    fallback_my_drive = fallback / "MyDrive"

    print(f"[drive] Tentando montagem alternativa em {fallback} ...")
    drive.mount(str(fallback), force_remount=True)

    if not fallback_my_drive.exists():
        raise RuntimeError("Drive montado, mas MyDrive não foi encontrado.")

    print(f"[drive] Google Drive montado em: {fallback_my_drive}")
    return fallback_my_drive


MYDRIVE = mount_google_drive()

if MYDRIVE is None:
    # Fallback local, apenas para execução fora do Colab.
    MESTRADO_BASE = Path(".")
else:
    MESTRADO_BASE = MYDRIVE / "Mestrado"
os.environ["MESTRADO_BASE"] = str(MESTRADO_BASE)

print(f"[config] MESTRADO_BASE = {MESTRADO_BASE}")

# Caminhos canônicos.
KAGGLE_RAW_EVENTS_PATH = "/content/drive/MyDrive/Mestrado/02-datasets/02-processed/trace_raw_validated.parquet"
KAGGLE_AGG_SERIES_PATH = "/content/drive/MyDrive/Mestrado/02-datasets/03-features/window_5min_series.parquet"
KAGGLE_DIAG_OUTPUT_DIR = "/content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics"

# Exporta para a célula principal.
os.environ["KAGGLE_RAW_EVENTS_PATH"] = str(KAGGLE_RAW_EVENTS_PATH)
os.environ["KAGGLE_AGG_SERIES_PATH"] = str(KAGGLE_AGG_SERIES_PATH)
os.environ["KAGGLE_DIAG_OUTPUT_DIR"] = str(KAGGLE_DIAG_OUTPUT_DIR)

print("\n[config] Caminhos configurados:")
print(f"- KAGGLE_RAW_EVENTS_PATH = {KAGGLE_RAW_EVENTS_PATH}")
print(f"- KAGGLE_AGG_SERIES_PATH = {KAGGLE_AGG_SERIES_PATH}")
print(f"- KAGGLE_DIAG_OUTPUT_DIR = {KAGGLE_DIAG_OUTPUT_DIR}")

# Garante que continuam sendo objetos Path, mesmo depois de exportar para os.environ
KAGGLE_RAW_EVENTS_PATH = Path(KAGGLE_RAW_EVENTS_PATH)
KAGGLE_AGG_SERIES_PATH = Path(KAGGLE_AGG_SERIES_PATH)
KAGGLE_DIAG_OUTPUT_DIR = Path(KAGGLE_DIAG_OUTPUT_DIR)

print("\n[check] Existência dos arquivos:")
print(f"- RAW existe? {KAGGLE_RAW_EVENTS_PATH.exists()}")
print(f"- AGG existe? {KAGGLE_AGG_SERIES_PATH.exists()}")

if not KAGGLE_RAW_EVENTS_PATH.exists():
    raise FileNotFoundError(f"Arquivo RAW não encontrado: {KAGGLE_RAW_EVENTS_PATH}")

if not KAGGLE_AGG_SERIES_PATH.exists():
    raise FileNotFoundError(f"Arquivo agregado não encontrado: {KAGGLE_AGG_SERIES_PATH}")

KAGGLE_DIAG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n[ok] Configuração pronta. Execute a célula principal do diagnóstico.")

[drive] Google Drive já disponível em: /content/drive/MyDrive
[config] MESTRADO_BASE = /content/drive/MyDrive/Mestrado

[config] Caminhos configurados:
- KAGGLE_RAW_EVENTS_PATH = /content/drive/MyDrive/Mestrado/02-datasets/02-processed/trace_raw_validated.parquet
- KAGGLE_AGG_SERIES_PATH = /content/drive/MyDrive/Mestrado/02-datasets/03-features/window_5min_series.parquet
- KAGGLE_DIAG_OUTPUT_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics

[check] Existência dos arquivos:
- RAW existe? True
- AGG existe? True

[ok] Configuração pronta. Execute a célula principal do diagnóstico.


In [ ]:
# 99_kaggle_reference_diagnostics_v1_0.py
# Artefato A — Diagnóstico Kaggle de Paridade para o ramo Borg 2019 FULL
# Autor: Sérgio H. C. Costa / PPCOMP-IFES
# Versão: v1.0

from __future__ import annotations

import os
import sys
import json
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Any

def _pip_install_if_needed(package_name: str, import_name: str) -> None:
    try:
        __import__(import_name)
        print(f"[deps] {package_name} já disponível ({import_name}).")
    except Exception:
        print(f"[deps] Instalando {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package_name])

_pip_install_if_needed("pandas", "pandas")
_pip_install_if_needed("pyarrow", "pyarrow")
_pip_install_if_needed("tabulate", "tabulate")

import pandas as pd
import numpy as np

SCENARIO_LABEL = "W5_K24_H12_P1_TRAIN_M2S"
ARTIFACT_PREFIX = "99_kaggle_reference"
WINDOW_MINUTES = 5
WINDOW_US = WINDOW_MINUTES * 60 * 1_000_000

RAW_EVENTS_PATH: Optional[str] = os.environ.get("KAGGLE_RAW_EVENTS_PATH") or None
AGG_SERIES_PATH: Optional[str] = os.environ.get("KAGGLE_AGG_SERIES_PATH") or None

DISCOVER_CANDIDATES = False
MESTRADO_BASE_ENV = os.environ.get("MESTRADO_BASE")

SEARCH_ROOTS = [
    ".",
    "./data",
    "./reports",
    "./reports_nb00",
    "./reports_nb01",
    "./reports_nb03",
    "./reports_nb16",
]

if MESTRADO_BASE_ENV:
    SEARCH_ROOTS.extend([
        MESTRADO_BASE_ENV,
        str(Path(MESTRADO_BASE_ENV) / "04-reports"),
        str(Path(MESTRADO_BASE_ENV) / "02-datasets"),
        str(Path(MESTRADO_BASE_ENV) / "02-datasets" / "02-processed"),
        str(Path(MESTRADO_BASE_ENV) / "02-datasets" / "03-features"),
    ])
else:
    SEARCH_ROOTS.extend([
        "/content/drive/MyDrive/Mestrado",
        "/content/drive/MyDrive/Mestrado/04-reports",
        "/content/drive/MyDrive/Mestrado/02-datasets",
        "/content/drive/MyDrive/Mestrado/02-datasets/02-processed",
        "/content/drive/MyDrive/Mestrado/02-datasets/03-features",
    ])

MAX_DISCOVERY_FILES = 5000

OUTPUT_DIR = Path(os.environ.get("KAGGLE_DIAG_OUTPUT_DIR", "./reports_99_kaggle_reference_diagnostics"))

if str(OUTPUT_DIR).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    raise RuntimeError(
        "OUTPUT_DIR aponta para /content/drive, mas o Google Drive não parece montado. "
        "Execute primeiro a célula de montagem/configuração."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_TYPE_NAMES = {
    0: "UNKNOWN",
    1: "SUBMIT",
    2: "QUEUE",
    3: "ENABLE",
    4: "EVICT",
    5: "FAIL",
    6: "FINISH",
    7: "KILL",
    8: "LOST",
    9: "UPDATE_PENDING",
    10: "UPDATE_RUNNING",
}
LIFECYCLE_EXCLUDED_NUMERIC = {9, 10}
LIFECYCLE_EXCLUDED_TEXT = {"UPDATE_PENDING", "UPDATE_RUNNING"}

def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def bytes_human(n: Optional[int]) -> str:
    if n is None:
        return ""
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    x = float(n)
    for u in units:
        if abs(x) < 1024 or u == units[-1]:
            return f"{x:.2f} {u}"
        x /= 1024
    return f"{x:.2f} TiB"

def read_table(path: Path, nrows: Optional[int] = None) -> pd.DataFrame:
    suffixes = "".join(path.suffixes).lower()
    if suffixes.endswith(".parquet"):
        return pd.read_parquet(path)
    if suffixes.endswith(".csv") or suffixes.endswith(".csv.gz"):
        return pd.read_csv(path, nrows=nrows)
    if suffixes.endswith(".json") or suffixes.endswith(".jsonl"):
        return pd.read_json(path, lines=suffixes.endswith(".jsonl"))
    raise ValueError(f"Formato não suportado: {path}")

def sample_columns(path: Path, nrows: int = 2000) -> Optional[List[str]]:
    try:
        if "".join(path.suffixes).lower().endswith(".parquet"):
            return list(pd.read_parquet(path).columns)
        if path.suffix.lower() in {".csv", ".gz"} or "".join(path.suffixes).lower().endswith(".csv.gz"):
            return list(pd.read_csv(path, nrows=nrows).columns)
    except Exception:
        return None
    return None

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def find_first_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    lower_map = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

def is_numeric_series(s: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(s)

def discover_files() -> pd.DataFrame:
    rows = []
    exts = {".parquet", ".csv", ".gz"}
    scanned = 0
    for root_str in SEARCH_ROOTS:
        root = Path(root_str)
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if scanned >= MAX_DISCOVERY_FILES:
                break
            if not p.is_file():
                continue
            if p.suffix.lower() not in exts and not "".join(p.suffixes).lower().endswith(".csv.gz"):
                continue
            scanned += 1
            name = p.name.lower()
            name_score = 0
            for token in ["borg", "kaggle", "event", "events", "validated", "clean", "agg", "series", "temporal", "nb00", "nb01", "nb03", "01_", "03_"]:
                if token in name:
                    name_score += 1
            cols = sample_columns(p)
            if cols is None:
                continue
            colset = {c.lower() for c in cols}
            has_raw_hint = bool(colset & {"time", "timestamp", "t_rel_us", "event", "type", "event_type", "failed", "machine_id", "collection_id"})
            has_agg_hint = bool(colset & {"bucket_id", "bucket_start_us", "n_events", "n_failed", "fail_rate"})
            if not (has_raw_hint or has_agg_hint or name_score >= 2):
                continue
            rows.append({
                "path": str(p),
                "size_bytes": p.stat().st_size,
                "size_human": bytes_human(p.stat().st_size),
                "name_score": name_score,
                "has_raw_hint": has_raw_hint,
                "has_agg_hint": has_agg_hint,
                "n_columns": len(cols),
                "columns": ",".join(cols[:60]),
            })
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["has_agg_hint", "has_raw_hint", "name_score", "size_bytes"], ascending=[False, False, False, False]).reset_index(drop=True)
    return out

def event_type_to_string(v: Any) -> str:
    if pd.isna(v):
        return "<NA>"
    if isinstance(v, (int, np.integer)):
        return EVENT_TYPE_NAMES.get(int(v), str(int(v)))
    if isinstance(v, (float, np.floating)) and float(v).is_integer():
        return EVENT_TYPE_NAMES.get(int(v), str(int(v)))
    return str(v).strip().upper()

def derive_event_columns(df: pd.DataFrame) -> Dict[str, Optional[str]]:
    return {
        "time_col": find_first_col(df, ["t_rel_us", "time", "timestamp", "timestamp_us", "start_time", "event_time"]),
        "type_col": find_first_col(df, ["type", "event_type", "event_type_id", "type_id"]),
        "event_col": find_first_col(df, ["event", "event_name", "event_type_name"]),
        "failed_col": find_first_col(df, ["failed", "is_failed", "failure", "is_failure", "n_failed"]),
        "bucket_col": find_first_col(df, ["bucket_id", "bucket", "window_id"]),
        "bucket_start_col": find_first_col(df, ["bucket_start_us", "window_start_us", "start_us"]),
        "n_events_col": find_first_col(df, ["n_events", "event_count", "n_events_all", "count"]),
        "n_failed_col": find_first_col(df, ["n_failed", "failed_count", "event_FAIL_count", "fail_count"]),
        "fail_rate_col": find_first_col(df, ["fail_rate", "fail_rate_primary", "failure_rate"]),
    }

def make_event_key(df: pd.DataFrame, cols: Dict[str, Optional[str]]) -> pd.Series:
    type_col = cols.get("type_col")
    event_col = cols.get("event_col")
    if type_col and type_col in df.columns:
        s = df[type_col]
        if is_numeric_series(s):
            return s.astype("Int64").astype(str).replace("<NA>", "<NA>")
        return s.map(event_type_to_string)
    if event_col and event_col in df.columns:
        return df[event_col].map(event_type_to_string)
    raise ValueError("Não foi possível identificar coluna de tipo/evento no dado bruto.")

def make_is_fail(df: pd.DataFrame, cols: Dict[str, Optional[str]], event_key: Optional[pd.Series] = None) -> pd.Series:
    failed_col = cols.get("failed_col")
    if failed_col and failed_col in df.columns and failed_col != cols.get("n_failed_col"):
        s = df[failed_col]
        if pd.api.types.is_bool_dtype(s):
            return s.fillna(False)
        if is_numeric_series(s):
            return s.fillna(0).astype(float) > 0
        return s.astype(str).str.upper().isin({"1", "TRUE", "T", "YES", "Y", "FAIL", "FAILED"})
    if event_key is None:
        event_key = make_event_key(df, cols)
    return event_key.astype(str).str.upper().isin({"5", "FAIL"})

def make_is_lifecycle(df: pd.DataFrame, cols: Dict[str, Optional[str]], event_key: pd.Series) -> pd.Series:
    type_col = cols.get("type_col")
    if type_col and type_col in df.columns and is_numeric_series(df[type_col]):
        return ~df[type_col].astype("Int64").isin(list(LIFECYCLE_EXCLUDED_NUMERIC)).fillna(False)
    return ~event_key.astype(str).str.upper().isin(LIFECYCLE_EXCLUDED_TEXT)

def build_raw_diagnostics(raw_df: pd.DataFrame):
    df = normalize_columns(raw_df)
    cols = derive_event_columns(df)
    event_key = make_event_key(df, cols)
    is_fail = make_is_fail(df, cols, event_key)
    is_lifecycle = make_is_lifecycle(df, cols, event_key)
    total = int(len(df))
    n_failed = int(is_fail.sum())
    n_lifecycle = int(is_lifecycle.sum())
    n_update = int((~is_lifecycle).sum())
    n_failed_lifecycle = int((is_fail & is_lifecycle).sum())
    type_counts = (
        pd.DataFrame({"event_key": event_key.astype(str)})
        .value_counts("event_key")
        .reset_index(name="n_events")
        .sort_values("event_key")
    )
    type_counts["event_name"] = type_counts["event_key"].map(lambda x: EVENT_TYPE_NAMES.get(int(x), x) if str(x).lstrip("-").isdigit() else x)
    type_counts["share"] = type_counts["n_events"] / max(total, 1)
    denom = pd.DataFrame([
        {"metric": "n_events_all", "value": total, "description": "Todos os eventos no dado Kaggle de referência"},
        {"metric": "n_events_lifecycle", "value": n_lifecycle, "description": "Eventos excluindo UPDATE_PENDING e UPDATE_RUNNING, se identificáveis"},
        {"metric": "n_update_pending_running", "value": n_update, "description": "Eventos UPDATE_PENDING/UPDATE_RUNNING, se identificáveis"},
        {"metric": "n_failed", "value": n_failed, "description": "Eventos FAIL"},
        {"metric": "raw_fail_share_all_events", "value": n_failed / total if total else np.nan, "description": "FAIL / todos os eventos"},
        {"metric": "raw_fail_share_lifecycle", "value": n_failed_lifecycle / n_lifecycle if n_lifecycle else np.nan, "description": "FAIL / lifecycle"},
    ])
    agg = None
    time_col = cols.get("time_col")
    bucket_col = cols.get("bucket_col")
    if bucket_col and bucket_col in df.columns:
        bucket_id = pd.to_numeric(df[bucket_col], errors="coerce").astype("Int64")
    elif time_col and time_col in df.columns:
        time_vals = pd.to_numeric(df[time_col], errors="coerce")
        min_time = int(time_vals.dropna().min()) if time_vals.notna().any() else 0
        t_rel = time_vals - min_time
        bucket_id = np.floor(t_rel / WINDOW_US).astype("Int64")
    else:
        bucket_id = None
    if bucket_id is not None:
        tmp = pd.DataFrame({
            "bucket_id": bucket_id,
            "n_events_all_unit": 1,
            "is_lifecycle": is_lifecycle.astype(int),
            "is_failed": is_fail.astype(int),
        }).dropna(subset=["bucket_id"])
        tmp["bucket_id"] = tmp["bucket_id"].astype(int)
        grouped = tmp.groupby("bucket_id", as_index=False).agg(
            n_events_all=("n_events_all_unit", "sum"),
            n_events_lifecycle=("is_lifecycle", "sum"),
            n_failed=("is_failed", "sum"),
        )
        if not grouped.empty:
            min_b, max_b = int(grouped["bucket_id"].min()), int(grouped["bucket_id"].max())
            grid = pd.DataFrame({"bucket_id": np.arange(min_b, max_b + 1, dtype=int)})
            agg = grid.merge(grouped, on="bucket_id", how="left")
            for c in ["n_events_all", "n_events_lifecycle", "n_failed"]:
                agg[c] = agg[c].fillna(0).astype(int)
            agg["is_empty_bucket"] = agg["n_events_all"].eq(0)
            agg["fail_rate_all_events"] = np.where(agg["n_events_all"] > 0, agg["n_failed"] / agg["n_events_all"], 0.0)
            agg["fail_rate_lifecycle"] = np.where(agg["n_events_lifecycle"] > 0, agg["n_failed"] / agg["n_events_lifecycle"], 0.0)
            agg["bucket_start_us"] = agg["bucket_id"] * WINDOW_US
    summary = {
        "created_at_utc": now_utc(),
        "scenario_label": SCENARIO_LABEL,
        "window_minutes": WINDOW_MINUTES,
        "detected_columns": cols,
        "n_rows_raw": total,
        "n_failed": n_failed,
        "n_events_lifecycle": n_lifecycle,
        "n_update_pending_running": n_update,
        "raw_fail_share_all_events": n_failed / total if total else None,
        "raw_fail_share_lifecycle": n_failed_lifecycle / n_lifecycle if n_lifecycle else None,
        "event_type_counts": dict(zip(type_counts["event_key"].astype(str), type_counts["n_events"].astype(int))),
        "agg_available": agg is not None,
    }
    if agg is not None:
        summary.update({
            "n_windows_continuous": int(len(agg)),
            "n_empty_buckets": int(agg["is_empty_bucket"].sum()),
            "fail_rate_all_events_min": float(agg["fail_rate_all_events"].min()),
            "fail_rate_all_events_mean": float(agg["fail_rate_all_events"].mean()),
            "fail_rate_all_events_std": float(agg["fail_rate_all_events"].std(ddof=0)),
            "fail_rate_all_events_max": float(agg["fail_rate_all_events"].max()),
            "fail_rate_lifecycle_min": float(agg["fail_rate_lifecycle"].min()),
            "fail_rate_lifecycle_mean": float(agg["fail_rate_lifecycle"].mean()),
            "fail_rate_lifecycle_std": float(agg["fail_rate_lifecycle"].std(ddof=0)),
            "fail_rate_lifecycle_max": float(agg["fail_rate_lifecycle"].max()),
        })
    return summary, type_counts, denom, agg

def build_agg_diagnostics(agg_df: pd.DataFrame):
    df = normalize_columns(agg_df)
    cols = derive_event_columns(df)
    fail_rate_col = cols.get("fail_rate_col")
    n_events_col = cols.get("n_events_col")
    n_failed_col = cols.get("n_failed_col")
    bucket_col = cols.get("bucket_col")
    summary: Dict[str, Any] = {
        "created_at_utc": now_utc(),
        "scenario_label": SCENARIO_LABEL,
        "window_minutes": WINDOW_MINUTES,
        "detected_columns": cols,
        "n_rows_agg": int(len(df)),
    }
    rows = []
    if n_events_col:
        rows.append({"metric": "sum_n_events", "value": float(pd.to_numeric(df[n_events_col], errors="coerce").sum())})
        summary["sum_n_events"] = int(pd.to_numeric(df[n_events_col], errors="coerce").sum())
    if n_failed_col:
        rows.append({"metric": "sum_n_failed", "value": float(pd.to_numeric(df[n_failed_col], errors="coerce").sum())})
        summary["sum_n_failed"] = int(pd.to_numeric(df[n_failed_col], errors="coerce").sum())
    if fail_rate_col:
        fr = pd.to_numeric(df[fail_rate_col], errors="coerce")
        desc = fr.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).reset_index()
        desc.columns = ["metric", "value"]
        desc["metric"] = "fail_rate_" + desc["metric"].astype(str)
        rows.extend(desc.to_dict("records"))
        summary.update({
            "fail_rate_col": fail_rate_col,
            "fail_rate_min": float(fr.min()),
            "fail_rate_mean": float(fr.mean()),
            "fail_rate_std": float(fr.std(ddof=0)),
            "fail_rate_max": float(fr.max()),
        })
    if bucket_col:
        b = pd.to_numeric(df[bucket_col], errors="coerce").dropna()
        if not b.empty:
            continuous = int(b.max() - b.min() + 1)
            rows.extend([
                {"metric": "bucket_min", "value": float(b.min())},
                {"metric": "bucket_max", "value": float(b.max())},
                {"metric": "n_buckets_continuous_span", "value": continuous},
                {"metric": "n_buckets_observed", "value": int(b.nunique())},
                {"metric": "n_missing_buckets_in_span", "value": int(continuous - b.nunique())},
            ])
            summary.update({
                "bucket_min": int(b.min()),
                "bucket_max": int(b.max()),
                "n_buckets_observed": int(b.nunique()),
                "n_missing_buckets_in_span": int(continuous - b.nunique()),
            })
    return summary, pd.DataFrame(rows)

def write_markdown(summary: Dict[str, Any], output_paths: Dict[str, str]) -> Path:
    md = []
    md.append("# Artefato A — Diagnóstico Kaggle de Paridade")
    md.append("")
    md.append(f"**Criado em:** {summary.get('created_at_utc')}")
    md.append(f"**Cenário de referência:** `{SCENARIO_LABEL}`")
    md.append("")
    md.append("## 1. Objetivo")
    md.append("")
    md.append("Este diagnóstico registra a definição efetiva de denominador e a distribuição de tipos de evento do cenário Kaggle canônico. O objetivo é permitir a paridade com o ramo Borg 2019 `_FULL` antes da geração do Parquet oficial.")
    md.append("")
    md.append("## 2. Resultados principais")
    md.append("")
    keys = [
        "raw_path", "agg_path", "n_rows_raw", "n_rows_agg", "n_failed", "n_events_lifecycle", "n_update_pending_running",
        "raw_fail_share_all_events", "raw_fail_share_lifecycle", "n_windows_continuous", "n_empty_buckets",
        "fail_rate_all_events_mean", "fail_rate_all_events_std", "fail_rate_all_events_max",
        "fail_rate_lifecycle_mean", "fail_rate_lifecycle_std", "fail_rate_lifecycle_max",
        "fail_rate_mean", "fail_rate_std", "fail_rate_max",
    ]
    md.append("| Métrica | Valor |")
    md.append("|---|---:|")
    for k in keys:
        if k in summary and summary[k] is not None:
            v = summary[k]
            vv = f"{v:.8f}" if isinstance(v, float) else str(v)
            md.append(f"| `{k}` | {vv} |")
    md.append("")
    md.append("## 3. Artefatos gerados")
    md.append("")
    for label, path in output_paths.items():
        md.append(f"- `{label}`: `{path}`")
    md.append("")
    md.append("## 4. Interpretação")
    md.append("")
    md.append("A escolha do denominador primário do ramo `_FULL` deve ser feita por comparação entre este diagnóstico e a paridade BigQuery. Este arquivo não autoriza, por si só, a reutilização do limiar global histórico do Kaggle nem o cálculo de limiar global retrospectivo no `_FULL`.")
    md.append("")
    md.append("## 5. Travas metodológicas")
    md.append("")
    md.append("1. O limiar `0.47840136`, se presente em artefatos históricos, permanece apenas como referência documental.")
    md.append("2. O ramo `_FULL` deve usar `TRAIN_M2S` estimado no trecho de treino, preferencialmente por célula.")
    md.append("3. O denominador `fail_rate_primary` deve reproduzir a definição efetiva do cenário canônico, conforme evidência de paridade.")
    md.append("")
    out = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_final.md"
    out.write_text("\n".join(md), encoding="utf-8")
    return out

def main() -> None:
    print("=" * 80)
    print("Artefato A — Diagnóstico Kaggle de Paridade")
    print("=" * 80)
    print(f"[config] OUTPUT_DIR = {OUTPUT_DIR}")

    raw_path = Path(RAW_EVENTS_PATH).expanduser() if RAW_EVENTS_PATH else None
    agg_path = Path(AGG_SERIES_PATH).expanduser() if AGG_SERIES_PATH else None

    if raw_path and not raw_path.exists():
        print(f"[warn] RAW_EVENTS_PATH não existe: {raw_path}")
        raw_path = None
    if agg_path and not agg_path.exists():
        print(f"[warn] AGG_SERIES_PATH não existe: {agg_path}")
        agg_path = None

    if DISCOVER_CANDIDATES and raw_path is None and agg_path is None:
        print("[discover] Buscando candidatos de arquivos Kaggle/artefatos canônicos...")
        candidates = discover_files()
        cand_path = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_file_candidates.csv"
        candidates.to_csv(cand_path, index=False)
        print(f"[discover] Candidatos salvos em: {cand_path}")
        if not candidates.empty:
            print(candidates[["path", "size_human", "has_raw_hint", "has_agg_hint", "name_score"]].head(20).to_markdown(index=False))
            for _, row in candidates.iterrows():
                p = Path(row["path"])
                try:
                    df_sample = read_table(p)
                    df_sample = normalize_columns(df_sample)
                    cols = derive_event_columns(df_sample)
                    if cols.get("type_col") or cols.get("event_col"):
                        raw_path = p
                        print(f"[discover] RAW_EVENTS_PATH selecionado automaticamente: {raw_path}")
                        break
                except Exception:
                    continue
            if raw_path is None:
                for _, row in candidates.iterrows():
                    p = Path(row["path"])
                    try:
                        df_sample = read_table(p)
                        df_sample = normalize_columns(df_sample)
                        cols = derive_event_columns(df_sample)
                        if cols.get("fail_rate_col") or (cols.get("n_events_col") and cols.get("n_failed_col")):
                            agg_path = p
                            print(f"[discover] AGG_SERIES_PATH selecionado automaticamente: {agg_path}")
                            break
                    except Exception:
                        continue

    summary: Dict[str, Any] = {"created_at_utc": now_utc(), "scenario_label": SCENARIO_LABEL, "artifact_prefix": ARTIFACT_PREFIX}
    output_paths: Dict[str, str] = {}

    if raw_path and raw_path.exists():
        print(f"[raw] Lendo dado bruto/validado: {raw_path}")
        raw_df = read_table(raw_path)
        raw_summary, type_counts, denom, agg = build_raw_diagnostics(raw_df)
        summary.update(raw_summary)
        summary["raw_path"] = str(raw_path)
        summary["raw_sha256"] = sha256_file(raw_path)

        p = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_event_type_counts.csv"
        type_counts.to_csv(p, index=False)
        output_paths["event_type_counts"] = str(p)

        p = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_denominator_summary.csv"
        denom.to_csv(p, index=False)
        output_paths["denominator_summary"] = str(p)

        if agg is not None:
            p = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_agg_5min_from_raw.parquet"
            agg.to_parquet(p, index=False)
            output_paths["agg_5min_from_raw"] = str(p)
            p2 = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_fail_rate_window_summary.csv"
            pd.DataFrame({
                "metric": [
                    "n_windows", "n_empty_buckets",
                    "fail_rate_all_events_mean", "fail_rate_all_events_std", "fail_rate_all_events_max",
                    "fail_rate_lifecycle_mean", "fail_rate_lifecycle_std", "fail_rate_lifecycle_max",
                ],
                "value": [
                    len(agg), int(agg["is_empty_bucket"].sum()),
                    agg["fail_rate_all_events"].mean(), agg["fail_rate_all_events"].std(ddof=0), agg["fail_rate_all_events"].max(),
                    agg["fail_rate_lifecycle"].mean(), agg["fail_rate_lifecycle"].std(ddof=0), agg["fail_rate_lifecycle"].max(),
                ]
            }).to_csv(p2, index=False)
            output_paths["fail_rate_window_summary"] = str(p2)

    elif agg_path and agg_path.exists():
        print(f"[agg] Lendo série agregada: {agg_path}")
        agg_df = read_table(agg_path)
        agg_summary, agg_metrics = build_agg_diagnostics(agg_df)
        summary.update(agg_summary)
        summary["agg_path"] = str(agg_path)
        summary["agg_sha256"] = sha256_file(agg_path)
        p = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_agg_existing_summary.csv"
        agg_metrics.to_csv(p, index=False)
        output_paths["agg_existing_summary"] = str(p)
    else:
        raise FileNotFoundError(
            "Não encontrei RAW_EVENTS_PATH nem AGG_SERIES_PATH. "
            "Defina uma dessas variáveis no topo do script/notebook ou pelas variáveis de ambiente "
            "KAGGLE_RAW_EVENTS_PATH / KAGGLE_AGG_SERIES_PATH."
        )

    p = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_summary.json"
    p.write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    output_paths["summary_json"] = str(p)

    manifest_rows = []
    for label, path_str in output_paths.items():
        pp = Path(path_str)
        if pp.exists():
            manifest_rows.append({
                "label": label,
                "path": str(pp),
                "size_bytes": pp.stat().st_size,
                "sha256": sha256_file(pp),
                "created_at_utc": now_utc(),
            })
    manifest = pd.DataFrame(manifest_rows)
    pman = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_artifact_manifest_sha256.csv"
    manifest.to_csv(pman, index=False)
    output_paths["artifact_manifest_sha256"] = str(pman)

    pmd = write_markdown(summary, output_paths)
    output_paths["final_md"] = str(pmd)

    print("\n[done] Diagnóstico Kaggle concluído.")
    print(f"[done] Saída: {OUTPUT_DIR}")
    print(f"[done] Final MD: {pmd}")
    print("\nResumo:")
    for k in ["n_rows_raw", "n_rows_agg", "n_failed", "n_events_lifecycle", "raw_fail_share_all_events", "raw_fail_share_lifecycle", "n_windows_continuous", "n_empty_buckets"]:
        if k in summary:
            print(f"- {k}: {summary[k]}")

if __name__ == "__main__":
    main()


[deps] pandas já disponível (pandas).
[deps] pyarrow já disponível (pyarrow).
[deps] tabulate já disponível (tabulate).
Artefato A — Diagnóstico Kaggle de Paridade
[config] OUTPUT_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics
[raw] Lendo dado bruto/validado: /content/drive/MyDrive/Mestrado/02-datasets/02-processed/trace_raw_validated.parquet

[done] Diagnóstico Kaggle concluído.
[done] Saída: /content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics
[done] Final MD: /content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics/99_kaggle_reference_final.md

Resumo:
- n_rows_raw: 405891
- n_failed: 92678
- n_events_lifecycle: 405779
- raw_fail_share_all_events: 0.22833223697987884
- raw_fail_share_lifecycle: 0.22839525948853934
- n_windows_continuous: 8930
- n_empty_buckets: 2


In [ ]:
# ============================================================
# NB99_A — A.1 Reconciliação contra a série canônica
# Reconstrução RAW x window_5min_series.parquet
#
# Reforços metodológicos:
# 1) compara n_janelas;
# 2) compara fail_rate linha a linha;
# 3) compara n_events_all e n_failed linha a linha, quando disponíveis;
# 4) compara bucket_key apenas quando a chave canônica é real, não fallback posicional;
# 5) registra explicitamente o status da reconciliação.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import json
import hashlib
import numpy as np
import pandas as pd

ARTIFACT_PREFIX = "99_kaggle_reference"
SCENARIO_LABEL = "W5_K24_H12_P1_TRAIN_M2S"
WINDOW_MINUTES = 5
EMPTY_BUCKET_POLICY = "zero_fill_primary; nan_skip_diagnostic"
STD_POLICY_PRIMARY = "ddof0_population"
TOL = 1e-9

OUTPUT_DIR = Path(os.environ["KAGGLE_DIAG_OUTPUT_DIR"])
CANONICAL_AGG_PATH = Path(os.environ["KAGGLE_AGG_SERIES_PATH"])
RECON_AGG_PATH = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_agg_5min_from_raw.parquet"
SUMMARY_PATH = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_summary.json"

assert CANONICAL_AGG_PATH.exists(), f"Série canônica não encontrada: {CANONICAL_AGG_PATH}"
assert RECON_AGG_PATH.exists(), f"Agregação reconstruída não encontrada: {RECON_AGG_PATH}"

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out

def pick_col(df: pd.DataFrame, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def canonical_to_standard(df: pd.DataFrame):
    """
    Normaliza a série canônica para colunas comparáveis.
    Retorna:
      std: DataFrame padronizado
      meta: metadados sobre quais colunas reais foram encontradas
    """
    df = normalize_cols(df)

    bucket_col = pick_col(df, [
        "bucket_id", "bucket", "window_id", "window_index", "bucket_id_rel"
    ])
    window_col = pick_col(df, [
        "window_start", "bucket_start", "bucket_start_us", "time_start", "dt_window"
    ])
    n_events_col = pick_col(df, [
        "n_events", "n_events_all", "event_count", "events_count", "count_events"
    ])
    n_failed_col = pick_col(df, [
        "n_failed", "fail_count", "failed_count", "n_fail", "count_fail"
    ])
    fail_rate_col = pick_col(df, [
        "fail_rate", "failure_rate", "criticality", "target_rate"
    ])

    std = pd.DataFrame()

    if bucket_col:
        std["bucket_key"] = pd.to_numeric(df[bucket_col], errors="coerce")
        bucket_key_is_real = True
    else:
        # Fallback apenas posicional, não deve ser usado como prova de chave.
        std["bucket_key"] = np.arange(len(df), dtype=int)
        bucket_key_is_real = False

    if window_col:
        std["window_start_original"] = df[window_col].astype(str)
    else:
        std["window_start_original"] = None

    if n_events_col:
        std["n_events_all"] = pd.to_numeric(df[n_events_col], errors="coerce")
    else:
        std["n_events_all"] = np.nan

    if n_failed_col:
        std["n_failed"] = pd.to_numeric(df[n_failed_col], errors="coerce")
    else:
        std["n_failed"] = np.nan

    if fail_rate_col:
        std["fail_rate"] = pd.to_numeric(df[fail_rate_col], errors="coerce")
    elif n_events_col and n_failed_col:
        std["fail_rate"] = np.where(
            std["n_events_all"] > 0,
            std["n_failed"] / std["n_events_all"],
            0.0
        )
    else:
        raise ValueError(
            "Não foi possível identificar fail_rate nem n_events/n_failed "
            "na série canônica."
        )

    std["source_order"] = np.arange(len(std), dtype=int)

    meta = {
        "canonical_bucket_col": bucket_col,
        "canonical_bucket_key_is_real": bucket_key_is_real,
        "canonical_window_col": window_col,
        "canonical_n_events_col": n_events_col,
        "canonical_n_failed_col": n_failed_col,
        "canonical_fail_rate_col": fail_rate_col,
    }

    return std.sort_values("source_order").reset_index(drop=True), meta

def recon_to_standard(df: pd.DataFrame):
    df = normalize_cols(df)

    required = ["bucket_id", "n_events_all", "n_failed", "fail_rate_all_events"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Colunas ausentes na reconstrução: {missing}")

    std = pd.DataFrame({
        "bucket_key": pd.to_numeric(df["bucket_id"], errors="coerce"),
        "n_events_all": pd.to_numeric(df["n_events_all"], errors="coerce"),
        "n_failed": pd.to_numeric(df["n_failed"], errors="coerce"),
        "fail_rate": pd.to_numeric(df["fail_rate_all_events"], errors="coerce"),
        "is_empty_bucket": df.get("is_empty_bucket", False),
        "source_order": np.arange(len(df), dtype=int),
    })

    meta = {
        "recon_bucket_col": "bucket_id",
        "recon_bucket_key_is_real": True,
        "recon_n_events_col": "n_events_all",
        "recon_n_failed_col": "n_failed",
        "recon_fail_rate_col": "fail_rate_all_events",
    }

    return std.sort_values("source_order").reset_index(drop=True), meta

canonical_raw = pd.read_parquet(CANONICAL_AGG_PATH)
recon_raw = pd.read_parquet(RECON_AGG_PATH)

canonical, canonical_meta = canonical_to_standard(canonical_raw)
recon, recon_meta = recon_to_standard(recon_raw)

n_compare = min(len(canonical), len(recon))

cmp = pd.DataFrame({
    "row_pos": np.arange(n_compare, dtype=int),

    "canonical_bucket_key": canonical.loc[:n_compare-1, "bucket_key"].to_numpy(),
    "recon_bucket_key": recon.loc[:n_compare-1, "bucket_key"].to_numpy(),

    "canonical_n_events_all": canonical.loc[:n_compare-1, "n_events_all"].to_numpy(),
    "recon_n_events_all": recon.loc[:n_compare-1, "n_events_all"].to_numpy(),

    "canonical_n_failed": canonical.loc[:n_compare-1, "n_failed"].to_numpy(),
    "recon_n_failed": recon.loc[:n_compare-1, "n_failed"].to_numpy(),

    "canonical_fail_rate": canonical.loc[:n_compare-1, "fail_rate"].to_numpy(),
    "recon_fail_rate": recon.loc[:n_compare-1, "fail_rate"].to_numpy(),
})

cmp["diff_fail_rate"] = cmp["recon_fail_rate"] - cmp["canonical_fail_rate"]
cmp["abs_diff_fail_rate"] = cmp["diff_fail_rate"].abs()

cmp["diff_n_events"] = cmp["recon_n_events_all"] - cmp["canonical_n_events_all"]
cmp["abs_diff_n_events"] = cmp["diff_n_events"].abs()

cmp["diff_n_failed"] = cmp["recon_n_failed"] - cmp["canonical_n_failed"]
cmp["abs_diff_n_failed"] = cmp["diff_n_failed"].abs()

# Disponibilidade das contagens no canônico
counts_available = (
    canonical["n_events_all"].notna().all()
    and canonical["n_failed"].notna().all()
)

if counts_available:
    pass_counts = bool(
        (cmp["abs_diff_n_events"] == 0).all()
        and (cmp["abs_diff_n_failed"] == 0).all()
    )
    max_abs_diff_n_events = float(cmp["abs_diff_n_events"].max())
    max_abs_diff_n_failed = float(cmp["abs_diff_n_failed"].max())
    n_count_rows_match = int(
        ((cmp["abs_diff_n_events"] == 0) & (cmp["abs_diff_n_failed"] == 0)).sum()
    )
    count_row_match_ratio = float(
        ((cmp["abs_diff_n_events"] == 0) & (cmp["abs_diff_n_failed"] == 0)).mean()
    )
else:
    pass_counts = True
    max_abs_diff_n_events = None
    max_abs_diff_n_failed = None
    n_count_rows_match = None
    count_row_match_ratio = None

# Bucket key só entra como critério se a chave canônica for real.
bucket_key_comparable = bool(
    canonical_meta["canonical_bucket_key_is_real"]
    and recon_meta["recon_bucket_key_is_real"]
)

if bucket_key_comparable:
    cmp["bucket_key_match"] = cmp["canonical_bucket_key"] == cmp["recon_bucket_key"]
    pass_bucket_key = bool(cmp["bucket_key_match"].all())
    bucket_key_match_ratio = float(cmp["bucket_key_match"].mean())
else:
    cmp["bucket_key_match"] = np.nan
    pass_bucket_key = True
    bucket_key_match_ratio = None

# Estatísticas com e sem buckets vazios na reconstrução
fr_zero = recon["fail_rate"]
non_empty = recon["n_events_all"] > 0
fr_non_empty = recon.loc[non_empty, "fail_rate"]

same_n = len(canonical) == len(recon)
max_diff = float(cmp["abs_diff_fail_rate"].max()) if n_compare else None
pass_fail_rate = (max_diff is not None) and (max_diff <= TOL)

canonical_reconciliation_status = (
    "PASS"
    if same_n and pass_fail_rate and pass_counts and pass_bucket_key
    else "REVIEW_REQUIRED"
)

reconciliation_metrics = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "canonical_agg_path": str(CANONICAL_AGG_PATH),
    "canonical_agg_sha256": sha256_file(CANONICAL_AGG_PATH),
    "reconstructed_agg_path": str(RECON_AGG_PATH),
    "reconstructed_agg_sha256_before_metadata_update": sha256_file(RECON_AGG_PATH),

    "n_windows_canonical": int(len(canonical)),
    "n_windows_reconstructed": int(len(recon)),
    "n_rows_compared_positionally": int(n_compare),

    "canonical_bucket_col": canonical_meta["canonical_bucket_col"],
    "canonical_bucket_key_is_real": bool(canonical_meta["canonical_bucket_key_is_real"]),
    "bucket_key_comparable": bool(bucket_key_comparable),
    "bucket_key_row_match": bool(pass_bucket_key),
    "bucket_key_match_ratio": bucket_key_match_ratio,

    "counts_available": bool(counts_available),
    "counts_row_match": bool(pass_counts),
    "n_count_rows_match": n_count_rows_match,
    "count_row_match_ratio": count_row_match_ratio,
    "max_abs_diff_n_events": max_abs_diff_n_events,
    "max_abs_diff_n_failed": max_abs_diff_n_failed,

    "n_empty_buckets_reconstructed": int((recon["n_events_all"] == 0).sum()),

    "sum_n_events_canonical": None if canonical["n_events_all"].isna().all()
    else float(canonical["n_events_all"].sum()),
    "sum_n_events_reconstructed": float(recon["n_events_all"].sum()),

    "sum_n_failed_canonical": None if canonical["n_failed"].isna().all()
    else float(canonical["n_failed"].sum()),
    "sum_n_failed_reconstructed": float(recon["n_failed"].sum()),

    "max_abs_diff_fail_rate": max_diff,
    "mean_abs_diff_fail_rate": float(cmp["abs_diff_fail_rate"].mean()) if n_compare else None,
    "n_fail_rate_matches_tol": int((cmp["abs_diff_fail_rate"] <= TOL).sum()) if n_compare else 0,
    "fail_rate_match_ratio_tol": float((cmp["abs_diff_fail_rate"] <= TOL).mean()) if n_compare else None,

    "empty_bucket_policy": EMPTY_BUCKET_POLICY,
    "std_policy_primary": STD_POLICY_PRIMARY,

    "recon_fail_rate_mean_zero_fill": float(fr_zero.mean()),
    "recon_fail_rate_std_ddof0_zero_fill": float(fr_zero.std(ddof=0)),
    "recon_fail_rate_std_ddof1_zero_fill": float(fr_zero.std(ddof=1)),
    "recon_threshold_mu_2sigma_ddof0_zero_fill": float(fr_zero.mean() + 2 * fr_zero.std(ddof=0)),
    "recon_threshold_mu_2sigma_ddof1_zero_fill": float(fr_zero.mean() + 2 * fr_zero.std(ddof=1)),

    "recon_fail_rate_mean_non_empty": float(fr_non_empty.mean()),
    "recon_fail_rate_std_ddof0_non_empty": float(fr_non_empty.std(ddof=0)),
    "recon_threshold_mu_2sigma_ddof0_non_empty": float(
        fr_non_empty.mean() + 2 * fr_non_empty.std(ddof=0)
    ),

    "canonical_reconciliation_status": canonical_reconciliation_status,
}

recon_csv = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_canonical_reconciliation.csv"
pd.DataFrame([reconciliation_metrics]).to_csv(recon_csv, index=False)

mismatch_csv = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_canonical_reconciliation_mismatches.csv"

mismatch_mask = (
    (cmp["abs_diff_fail_rate"] > TOL)
    | (counts_available and (
        (cmp["abs_diff_n_events"] != 0)
        | (cmp["abs_diff_n_failed"] != 0)
    ))
)

if bucket_key_comparable:
    mismatch_mask = mismatch_mask | (~cmp["bucket_key_match"])

cmp.loc[mismatch_mask].head(500).to_csv(mismatch_csv, index=False)

# Atualiza summary_json
with SUMMARY_PATH.open("r", encoding="utf-8") as f:
    summary = json.load(f)

summary["canonical_reconciliation"] = reconciliation_metrics
summary["fail_numeric_type_confirmed_in_kaggle"] = False
summary["fail_numeric_type_note"] = (
    "O arquivo Kaggle validado não contém type_col numérico detectado; "
    "FAIL foi identificado por event/failed. A equivalência type=5→FAIL "
    "deve ser confirmada no NB99_B/BigQuery."
)

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("[ok] Reconciliação canônica concluída.")
print(f"- status: {canonical_reconciliation_status}")
print(f"- same_n: {same_n}")
print(f"- pass_fail_rate: {pass_fail_rate}")
print(f"- counts_available: {counts_available}")
print(f"- counts_row_match: {pass_counts}")
print(f"- bucket_key_comparable: {bucket_key_comparable}")
print(f"- bucket_key_row_match: {pass_bucket_key}")
print(f"- max_abs_diff_fail_rate: {max_diff}")
print(f"- max_abs_diff_n_events: {max_abs_diff_n_events}")
print(f"- max_abs_diff_n_failed: {max_abs_diff_n_failed}")
print(f"- arquivo: {recon_csv}")
print(f"- mismatches: {mismatch_csv}")

[ok] Reconciliação canônica concluída.
- status: PASS
- same_n: True
- pass_fail_rate: True
- counts_available: True
- counts_row_match: True
- bucket_key_comparable: True
- bucket_key_row_match: True
- max_abs_diff_fail_rate: 0.0
- max_abs_diff_n_events: 0.0
- max_abs_diff_n_failed: 0.0
- arquivo: /content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics/99_kaggle_reference_canonical_reconciliation.csv
- mismatches: /content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics/99_kaggle_reference_canonical_reconciliation_mismatches.csv


In [ ]:
# ============================================================
# NB99_A — Atualiza Parquet reconstruído com barreiras de procedência
# Executar depois da reconciliação canônica e antes do manifesto.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

ARTIFACT_PREFIX = "99_kaggle_reference"
SCENARIO_LABEL = "W5_K24_H12_P1_TRAIN_M2S"
WINDOW_MINUTES = 5
EMPTY_BUCKET_POLICY = "zero_fill_primary; nan_skip_diagnostic"
STD_POLICY_PRIMARY = "ddof0_population"

OUTPUT_DIR = Path(os.environ.get(
    "KAGGLE_DIAG_OUTPUT_DIR",
    "/content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics"
))

RECON_AGG_PATH = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_agg_5min_from_raw.parquet"

if not RECON_AGG_PATH.exists():
    raise FileNotFoundError(f"Parquet reconstruído não encontrado: {RECON_AGG_PATH}")

df_recon = pd.read_parquet(RECON_AGG_PATH)

created_at_utc = datetime.now(timezone.utc).isoformat()

df_recon["scenario_label"] = SCENARIO_LABEL
df_recon["artifact_prefix"] = ARTIFACT_PREFIX
df_recon["created_at_utc"] = created_at_utc
df_recon["source_kind"] = "kaggle_reference_reconstructed_from_raw"
df_recon["window_minutes"] = WINDOW_MINUTES
df_recon["empty_bucket_policy"] = EMPTY_BUCKET_POLICY
df_recon["std_policy_primary"] = STD_POLICY_PRIMARY

table = pa.Table.from_pandas(df_recon, preserve_index=False)

metadata = dict(table.schema.metadata or {})
metadata.update({
    b"scenario_label": SCENARIO_LABEL.encode("utf-8"),
    b"artifact_prefix": ARTIFACT_PREFIX.encode("utf-8"),
    b"created_at_utc": created_at_utc.encode("utf-8"),
    b"source_kind": b"kaggle_reference_reconstructed_from_raw",
    b"window_minutes": str(WINDOW_MINUTES).encode("utf-8"),
    b"empty_bucket_policy": EMPTY_BUCKET_POLICY.encode("utf-8"),
    b"std_policy_primary": STD_POLICY_PRIMARY.encode("utf-8"),
})

table = table.replace_schema_metadata(metadata)
pq.write_table(table, RECON_AGG_PATH)

print("[ok] Parquet atualizado com colunas e metadata de procedência.")
print(f"- {RECON_AGG_PATH}")
print(f"- linhas: {len(df_recon)}")
print(f"- colunas adicionadas/verificadas: scenario_label, artifact_prefix, created_at_utc, source_kind, window_minutes, empty_bucket_policy, std_policy_primary")

[ok] Parquet atualizado com colunas e metadata de procedência.
- /content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics/99_kaggle_reference_agg_5min_from_raw.parquet
- linhas: 8930
- colunas adicionadas/verificadas: scenario_label, artifact_prefix, created_at_utc, source_kind, window_minutes, empty_bucket_policy, std_policy_primary


In [ ]:
# ============================================================
# NB99_A — Escrita do Markdown final
# Deve ser executado antes do manifesto final.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import pandas as pd

ARTIFACT_PREFIX = "99_kaggle_reference"

OUTPUT_DIR = Path(os.environ.get(
    "KAGGLE_DIAG_OUTPUT_DIR",
    "/content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics"
))

RECON_PATH = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_canonical_reconciliation.csv"
FINAL_MD_PATH = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_final.md"

if not RECON_PATH.exists():
    raise FileNotFoundError(f"Reconciliação canônica não encontrada: {RECON_PATH}")

recon = pd.read_csv(RECON_PATH).iloc[0].to_dict()

final_md = f"""# 99_A. Conclusão da Etapa — Diagnóstico Kaggle de Paridade

## 99_A.1 Papel da etapa

O NB99_A foi executado como etapa auxiliar de diagnóstico do cenário Kaggle canônico `W5_K24_H12_P1_TRAIN_M2S`. Seu objetivo não foi recalcular modelos, alterar a formulação supervisionada ou reabrir o pipeline consolidado da dissertação, mas registrar, de forma auditável, a definição efetiva de denominador da taxa de falha e a distribuição dos tipos de evento na amostra Kaggle utilizada como referência.

Essa etapa cumpre papel preparatório para o ramo Borg 2019 `_FULL`, pois fornece a base empírica necessária para comparar a agregação BigQuery do dataset completo com o cenário canônico já consolidado. Dessa forma, o NB99_A preserva a separação entre os dois mundos do projeto: o cenário Kaggle permanece como base da dissertação, enquanto o cenário `_FULL` é tratado como ramo de replicação ampliada e validação de escala.

## 99_A.2 Reconciliação canônica

Além de reconstruir a série de 5 minutos a partir do RAW validado, o NB99_A reconciliou essa reconstrução contra a série canônica `window_5min_series.parquet`, isto é, contra a série efetivamente usada no pipeline da dissertação.

Resultado da reconciliação:

| Métrica | Valor |
|---|---:|
| Status | `{recon.get("canonical_reconciliation_status")}` |
| Janelas na série canônica | {int(recon.get("n_windows_canonical"))} |
| Janelas na reconstrução | {int(recon.get("n_windows_reconstructed"))} |
| Eventos na série canônica | {int(recon.get("sum_n_events_canonical"))} |
| Eventos na reconstrução | {int(recon.get("sum_n_events_reconstructed"))} |
| Falhas na série canônica | {int(recon.get("sum_n_failed_canonical"))} |
| Falhas na reconstrução | {int(recon.get("sum_n_failed_reconstructed"))} |
| Diferença máxima de `fail_rate` | {float(recon.get("max_abs_diff_fail_rate")):.12f} |
| Proporção de janelas compatíveis na tolerância | {float(recon.get("fail_rate_match_ratio_tol")):.8f} |

A reconciliação confirma que a reconstrução diagnóstica está alinhada com a série canônica da dissertação não apenas quanto à escala de `fail_rate`, mas também quanto às contagens linha a linha de eventos e falhas, quando fornecidas pela série canônica.

## 99_A.3 Política de buckets vazios e desvio-padrão

A política primária adotada para buckets vazios é `zero_fill_primary`: janelas sem eventos permanecem na série contínua com `n_events = 0`, `n_failed = 0` e `fail_rate = 0`. Como diagnóstico complementar, também foram calculadas estatísticas excluindo buckets vazios.

A política primária de desvio-padrão registrada é `ddof0_population`. Também foi registrado o valor com `ddof=1` para auditoria.

| Métrica | Valor |
|---|---:|
| Buckets vazios reconstruídos | {int(recon.get("n_empty_buckets_reconstructed"))} |
| Média `fail_rate` com zero-fill | {float(recon.get("recon_fail_rate_mean_zero_fill")):.8f} |
| Std `ddof=0` com zero-fill | {float(recon.get("recon_fail_rate_std_ddof0_zero_fill")):.8f} |
| Std `ddof=1` com zero-fill | {float(recon.get("recon_fail_rate_std_ddof1_zero_fill")):.8f} |
| μ+2σ `ddof=0` com zero-fill | {float(recon.get("recon_threshold_mu_2sigma_ddof0_zero_fill")):.8f} |
| μ+2σ `ddof=1` com zero-fill | {float(recon.get("recon_threshold_mu_2sigma_ddof1_zero_fill")):.8f} |
| μ+2σ `ddof=0` sem buckets vazios | {float(recon.get("recon_threshold_mu_2sigma_ddof0_non_empty")):.8f} |
| Contagens linha a linha disponíveis | {recon.get("counts_available")} |
| Contagens linha a linha compatíveis | {recon.get("counts_row_match")} |
| Diferença máxima de `n_events` | {float(recon.get("max_abs_diff_n_events") or 0):.0f} |
| Diferença máxima de `n_failed` | {float(recon.get("max_abs_diff_n_failed") or 0):.0f} |
| Chave de bucket comparável | {recon.get("bucket_key_comparable")} |
| Chave de bucket compatível, quando aplicável | {recon.get("bucket_key_row_match")} |

Esses números explicam a proximidade entre a reconstrução diagnóstica e o limiar retrospectivo histórico do cenário Kaggle. Entretanto, esse limiar permanece apenas como referência documental e não deve ser reutilizado no ramo `_FULL`.

## 99_A.4 Interpretação metodológica

O diagnóstico confirma que, no cenário Kaggle canônico, a escolha entre `FAIL / all_events` e `FAIL / lifecycle` não altera de forma material a escala da taxa de falha. Essa constatação não autoriza, contudo, a antecipar a mesma equivalência no Borg 2019 completo, pois o volume relativo de `UPDATE_PENDING` e `UPDATE_RUNNING` pode ser substancialmente diferente nas oito células do trace completo.

Assim, a decisão sobre o `fail_rate_primary` do ramo `_FULL` deve permanecer condicionada à paridade BigQuery. O NB99_A fornece o lado Kaggle da comparação; o NB99_B deverá produzir o lado Borg 2019 completo, por célula, permitindo avaliar a compatibilidade de denominadores e escalas.

## 99_A.5 Limitação sobre o mapeamento numérico de FAIL

O mapeamento numérico `type = 5 → FAIL` não é confirmado pelo NB99_A, pois o arquivo Kaggle validado não contém coluna numérica `type` detectada. No Kaggle, `FAIL` foi identificado por coluna textual/booleana (`event`/`failed`). A confirmação do mapeamento numérico ficará atribuída ao NB99_B, no lado BigQuery/FULL, por meio do histograma de `type` numérico e do respectivo crosswalk.

## 99_A.6 Travas preservadas

A execução do NB99_A preserva três travas metodológicas fundamentais:

1. o limiar histórico `0.47840136`, quando presente em artefatos anteriores, permanece apenas como referência documental do cenário Kaggle;
2. o ramo `_FULL` deverá utilizar `TRAIN_M2S`, estimado exclusivamente no trecho de treino, preferencialmente por célula;
3. o denominador `fail_rate_primary` do `_FULL` deverá ser escolhido por evidência de paridade, e não por conveniência operacional.

## 99_A.7 Encaminhamento

O Artefato A está aprovado como diagnóstico Kaggle de paridade, condicionado à preservação dos artefatos de reconciliação canônica e do manifesto final. A próxima etapa deve ser o NB99_B, responsável pela paridade BigQuery do Borg 2019 completo, ainda sem execução da query suficiente definitiva.

O objetivo do NB99_B será comparar, por célula, a distribuição de tipos, os denominadores, a escala da taxa de falha, a continuidade temporal, o mapeamento `type = 5 → FAIL` e a proximidade com o cenário Kaggle canônico.

---

Criado/atualizado em: {datetime.now(timezone.utc).isoformat()}
"""

FINAL_MD_PATH.write_text(final_md, encoding="utf-8")

print("[ok] Markdown final atualizado.")
print(f"- {FINAL_MD_PATH}")
print(f"- caracteres: {len(final_md)}")

[ok] Markdown final atualizado.
- /content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics/99_kaggle_reference_final.md
- caracteres: 5432


In [ ]:
# ============================================================
# NB99_A — Manifesto final único
# Deve ser executado APENAS no final, depois de:
# 1) execução principal;
# 2) reconciliação canônica;
# 3) atualização do Parquet com procedência;
# 4) geração/atualização do final.md.
#
# O manifesto NÃO inclui a si mesmo para evitar auto-referência.
# O hash do manifesto é registrado em arquivo meta separado.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import hashlib
import json
import pandas as pd

OUTPUT_DIR = Path(os.environ.get(
    "KAGGLE_DIAG_OUTPUT_DIR",
    "/content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics"
))

ARTIFACT_PREFIX = "99_kaggle_reference"

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

# Artefatos principais do NB99_A.
# O manifesto lista os artefatos finais, mas NÃO inclui o próprio manifesto
# nem o manifest_meta, para evitar circularidade.
artifact_specs = [
    ("event_type_counts", f"{ARTIFACT_PREFIX}_event_type_counts.csv"),
    ("denominator_summary", f"{ARTIFACT_PREFIX}_denominator_summary.csv"),
    ("agg_5min_from_raw", f"{ARTIFACT_PREFIX}_agg_5min_from_raw.parquet"),
    ("fail_rate_window_summary", f"{ARTIFACT_PREFIX}_fail_rate_window_summary.csv"),

    # Novos artefatos da etapa A.1
    ("canonical_reconciliation", f"{ARTIFACT_PREFIX}_canonical_reconciliation.csv"),
    ("canonical_reconciliation_mismatches", f"{ARTIFACT_PREFIX}_canonical_reconciliation_mismatches.csv"),

    # Artefatos textuais e de síntese
    ("summary_json", f"{ARTIFACT_PREFIX}_summary.json"),
    ("final_md", f"{ARTIFACT_PREFIX}_final.md"),
]

records = []

for role, filename in artifact_specs:
    path = OUTPUT_DIR / filename

    if not path.exists():
        print(f"[warn] Artefato ausente no manifesto final: {filename}")
        continue

    stat = path.stat()

    records.append({
        "artifact_role": role,
        "filename": filename,
        "relative_path": filename,
        "absolute_path": str(path),
        "size_bytes": stat.st_size,
        "sha256": sha256_file(path),
        "modified_at_utc": datetime.fromtimestamp(
            stat.st_mtime, tz=timezone.utc
        ).isoformat(),
    })

manifest_df = (
    pd.DataFrame(records)
    .sort_values(["artifact_role", "filename"])
    .reset_index(drop=True)
)

manifest_path = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_artifact_manifest_sha256.csv"
manifest_df.to_csv(manifest_path, index=False, encoding="utf-8")

manifest_sha256 = sha256_file(manifest_path)

manifest_meta = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "artifact_prefix": ARTIFACT_PREFIX,
    "manifest_filename": manifest_path.name,
    "manifest_sha256": manifest_sha256,
    "manifest_policy": (
        "O manifesto lista os artefatos principais finais do NB99_A, "
        "incluindo summary_json, final_md e os artefatos de reconciliação "
        "canônica. O manifesto não inclui a si mesmo nem o manifest_meta "
        "para evitar auto-referência. O hash do manifesto é registrado "
        "neste arquivo separado de metadados."
    ),
    "n_manifested_artifacts": int(len(manifest_df)),
    "manifested_artifacts": manifest_df["filename"].tolist(),
}

manifest_meta_path = OUTPUT_DIR / f"{ARTIFACT_PREFIX}_artifact_manifest_meta.json"

with manifest_meta_path.open("w", encoding="utf-8") as f:
    json.dump(manifest_meta, f, indent=2, ensure_ascii=False)

print("[ok] Manifesto final do NB99_A gerado.")
print(f"- Manifesto: {manifest_path}")
print(f"- SHA-256 do manifesto: {manifest_sha256}")
print(f"- Meta: {manifest_meta_path}")
print(f"- Artefatos no manifesto: {len(manifest_df)}")

display(manifest_df)

[ok] Manifesto final do NB99_A gerado.
- Manifesto: /content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics/99_kaggle_reference_artifact_manifest_sha256.csv
- SHA-256 do manifesto: 8336626a43684c22af94415332b974887fc45258298c0601d4fd8b03ee9b4a71
- Meta: /content/drive/MyDrive/Mestrado/04-reports/99_A_kaggle_diagnostics/99_kaggle_reference_artifact_manifest_meta.json
- Artefatos no manifesto: 8


,artifact_role,filename,relative_path,absolute_path,size_bytes,sha256,modified_at_utc
0,agg_5min_from_raw,99_kaggle_reference_agg_5min_from_raw.parquet,99_kaggle_reference_agg_5min_from_raw.parquet,/content/drive/MyDrive/Mestrado/04-reports/99_...,200773,87e735f98aa68914d843f57e6848ec1ed804d1f8c784a2...,2026-06-18T23:40:40+00:00
1,canonical_reconciliation,99_kaggle_reference_canonical_reconciliation.csv,99_kaggle_reference_canonical_reconciliation.csv,/content/drive/MyDrive/Mestrado/04-reports/99_...,1729,8d66329d9bb96a47d1bf4c2bc0b7763ebda6811a283aa3...,2026-06-18T23:40:40+00:00
2,canonical_reconciliation_mismatches,99_kaggle_reference_canonical_reconciliation_m...,99_kaggle_reference_canonical_reconciliation_m...,/content/drive/MyDrive/Mestrado/04-reports/99_...,273,b44786cab821703a98e90ad436ae1da227e8f3296567f1...,2026-06-18T23:40:40+00:00
3,denominator_summary,99_kaggle_reference_denominator_summary.csv,99_kaggle_reference_denominator_summary.csv,/content/drive/MyDrive/Mestrado/04-reports/99_...,447,54897a6ab7def24b28f09a3dbab687a6a9061ddf0fdcf2...,2026-06-18T23:40:40+00:00
4,event_type_counts,99_kaggle_reference_event_type_counts.csv,99_kaggle_reference_event_type_counts.csv,/content/drive/MyDrive/Mestrado/04-reports/99_...,451,813231d1f4cd6ca7abcb2585602362d16c4b167a2ab7ee...,2026-06-18T23:40:40+00:00
5,fail_rate_window_summary,99_kaggle_reference_fail_rate_window_summary.csv,99_kaggle_reference_fail_rate_window_summary.csv,/content/drive/MyDrive/Mestrado/04-reports/99_...,286,c159685d3b8c3c9b831e925b8b323161688971080dcfc8...,2026-06-18T23:40:40+00:00
6,final_md,99_kaggle_reference_final.md,99_kaggle_reference_final.md,/content/drive/MyDrive/Mestrado/04-reports/99_...,5628,4ffc8b25d833cc787ac9059702c896e8994c18e1a097db...,2026-06-18T23:40:40+00:00
7,summary_json,99_kaggle_reference_summary.json,99_kaggle_reference_summary.json,/content/drive/MyDrive/Mestrado/04-reports/99_...,3848,1b8c3757fa0b0eb4d13f4a4aa8727dbbff9846dda3a13d...,2026-06-18T23:40:40+00:00


# 99_A. Conclusão da Etapa — Diagnóstico de Paridade do Protótipo Kaggle

## 99_A.1 Papel da etapa

O NB99_A foi executado como etapa auxiliar de diagnóstico do protótipo Kaggle refinado `W5_K24_H12_P1_TRAIN_M2S`. A etapa registrou, de forma auditável, a definição efetiva do denominador da taxa de falha e a distribuição dos tipos de evento na amostra utilizada como referência, sem interferir na formulação supervisionada ou nos resultados de modelagem.

Essa etapa cumpre papel preparatório para o experimento FULL, pois fornece a base empírica necessária para comparar a agregação BigQuery do dataset completo com o protótipo Kaggle refinado. Dessa forma, o NB99_A preserva a separação entre as duas camadas experimentais: o protótipo Kaggle refinado registra o desenvolvimento metodológico, enquanto o experimento FULL fornece a validação em escala.

## 99_A.2 Reconciliação com a série de referência

Além de reconstruir a série de 5 minutos a partir do RAW validado, o NB99_A reconciliou essa reconstrução contra a série de referência `window_5min_series.parquet`, isto é, contra a série efetivamente utilizada no protótipo Kaggle.

Resultado da reconciliação:

| Métrica | Valor |
|---|---:|
| Status | `PASS` |
| Janelas na série de referência | 8930 |
| Janelas na reconstrução | 8930 |
| Eventos na série de referência | 405891 |
| Eventos na reconstrução | 405891 |
| Falhas na série de referência | 92678 |
| Falhas na reconstrução | 92678 |
| Diferença máxima de `fail_rate` | 0.000000000000 |
| Proporção de janelas compatíveis na tolerância | 1.00000000 |

A reconciliação confirma que a reconstrução diagnóstica está alinhada com a série de referência do protótipo Kaggle não apenas quanto à escala de `fail_rate`, mas também quanto às contagens linha a linha de eventos e falhas, quando fornecidas pela série de referência.

## 99_A.3 Política de buckets vazios e desvio-padrão

A política primária adotada para buckets vazios é `zero_fill_primary`: janelas sem eventos permanecem na série contínua com `n_events = 0`, `n_failed = 0` e `fail_rate = 0`. Como diagnóstico complementar, também foram calculadas estatísticas excluindo buckets vazios.

A política primária de desvio-padrão registrada é `ddof0_population`. Também foi registrado o valor com `ddof=1` para auditoria.

| Métrica | Valor |
|---|---:|
| Buckets vazios reconstruídos | 2 |
| Média `fail_rate` com zero-fill | 0.19657957 |
| Std `ddof=0` com zero-fill | 0.14091055 |
| Std `ddof=1` com zero-fill | 0.14091844 |
| μ+2σ `ddof=0` com zero-fill | 0.47840068 |
| μ+2σ `ddof=1` com zero-fill | 0.47841646 |
| μ+2σ `ddof=0` sem buckets vazios | 0.47841483 |
| Contagens linha a linha disponíveis | True |
| Contagens linha a linha compatíveis | True |
| Diferença máxima de `n_events` | 0 |
| Diferença máxima de `n_failed` | 0 |
| Chave de bucket comparável | True |
| Chave de bucket compatível, quando aplicável | True |

Esses números explicam a proximidade entre a reconstrução diagnóstica e o limiar retrospectivo histórico do protótipo Kaggle. Entretanto, esse limiar permanece apenas como referência documental e não deve ser reutilizado no experimento FULL.

## 99_A.4 Interpretação metodológica

O diagnóstico confirma que, no protótipo Kaggle refinado, a escolha entre `FAIL / all_events` e `FAIL / lifecycle` não altera de forma material a escala da taxa de falha. Essa constatação não autoriza, contudo, a antecipar a mesma equivalência no Borg 2019 completo, pois o volume relativo de `UPDATE_PENDING` e `UPDATE_RUNNING` pode ser substancialmente diferente nas oito células do trace completo.

Assim, a definição do `fail_rate_primary` do experimento FULL é condicionada à paridade obtida no BigQuery. O NB99_A fornece a referência do protótipo Kaggle refinado, enquanto o NB99_B produz o diagnóstico correspondente do Borg 2019 completo por célula, permitindo avaliar a compatibilidade de denominadores e escalas.

## 99_A.5 Limitação sobre o mapeamento numérico de FAIL

O mapeamento numérico `type = 5 → FAIL` não é verificado pelo NB99_A, pois o arquivo do protótipo Kaggle não contém uma coluna numérica `type`. Nessa base, `FAIL` foi identificado pelas colunas textual e booleana (`event` e `failed`). O NB99_B verifica o mapeamento no lado BigQuery/FULL por meio do histograma de `type` numérico e do respectivo crosswalk.

## 99_A.6 Travas preservadas

A execução do NB99_A preserva três travas metodológicas fundamentais:

1. o limiar histórico `0.47840136`, quando presente em artefatos anteriores, permanece apenas como referência documental do protótipo Kaggle;
2. o experimento FULL utiliza `TRAIN_M2S`, estimado exclusivamente no trecho de treino e por célula;
3. o denominador `fail_rate_primary` do FULL é definido por evidência de paridade, e não por conveniência operacional.

## 99_A.7 Encaminhamento

O conjunto de artefatos do NB99_A constitui a referência de paridade do protótipo Kaggle refinado. A reconciliação com a série de referência e o manifesto preservam a rastreabilidade da etapa.

O NB99_B complementa essa evidência com a análise do Borg 2019 completo por célula, abrangendo distribuição de tipos, denominadores, escala da taxa de falha, continuidade temporal, mapeamento `type = 5 → FAIL` e proximidade com o protótipo Kaggle refinado.

<!-- FIM DA CÉLULA MARKDOWN DE CONCLUSÃO -->
